In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import pandas as pd

In [ ]:




# le lien de base avec la selection de tractors
base_url = "https://www.truckscout24.com/main/search/index?mainCategoryIds=252&subCategoryIds=&manufacturers=&models%5B0%5D=&priceTo=&usePriceGross=0&manufacturedFrom=&countries=&properties%5Baxle+configuration%5D%5Bvalue%5D="

# liste vide pour stocker les differents informations
dataset = []

# boucles pour parcourir les 8 pages
for i in range(1,9):

    if i == 1:
        # initialisation
        url = base_url
    else:
        # incrumentation
        url = base_url + "&page=" + str(i)

    # Parser le contenu HTML
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    print(" I - Je dors  ")
    # Attendre avant de continuer
    time.sleep(5)

    # Trouver tous les camions tractor sur la page
    tractors = soup.find_all("section", class_="shadow-sm grid-card grid-card-border")

    # parcourir chaque tractor sur la page
    for tractor in tractors:

        # dictionnaire vide pour recuillir les infos 
        truck = {}

        # l'identifiant du tractor
        truck["listing_id"] = tractor["data-listing-id"]

        # marque du tractor
        truck["manufacturer"] = tractor.find("h2").find_all("span")[1].text

        # le prix
        truck["price"] = tractor.find("div", class_="price").text.strip()

        # le pays/ville
        truck["location"] = tractor.find("div", class_="country-name").text.strip()

        # les autres infos sont accesible seulement avec un lien specifique à chaque tractor
        link = "https://www.truckscout24.com" + tractor.find("a")["href"]

        detail_page = requests.get(link)
        detail_soup = BeautifulSoup(detail_page.text, "html.parser")

        #print(" I - Je dors dans la page  ")
        # Attendre avant de continuer
        time.sleep(5)

        # definir les informations qui nous interessent
        liste_info = [
            "Year of construction:",
            "Condition:",
            "Mileage:",
            "Update:",
            "Location:",
            "Model:",
            "Listing ID:"
        ]

        # parcourir les infos
        for dl in detail_soup.find_all("dl"):

            dt = dl.find("dt")

            # verifier si l'info nous interessent
            if dt and dt.text.strip() in liste_info:

                dd = dl.find("dd")

                if dd:
                    # stocké la valeur
                    truck[dt.text.strip()] = dd.text.strip()

        dataset.append(truck)

df = pd.DataFrame(dataset)

 I - Je dors  
 I - Je dors  
 I - Je dors  
 I - Je dors  
 I - Je dors  
 I - Je dors  
 I - Je dors  
 I - Je dors  


In [54]:
df.head()

,listing_id,manufacturer,price,location,Model:,Condition:,Location:,Listing ID:,Update:,Year of construction:,Mileage:
0,21468897,Kenworth,"$43,000 \n ...",Atlanta,W900L,ready for operation (used),"Atlanta, United States",A214-68-897,10.03.2026,NaN,NaN
1,21507582,Scania,"€57,174 \n ...",Norway,R730 6x4 Tractor Unit with Hydraulics and Rais...,used,Trøndelag,A215-07-582,15.03.2026,2016,"700,000 km"
2,21486682,Volvo,"€147,616 \n ...",Norway,FH 500 6x4 Tractor w/ 2024 Rojo Tipper Semi,used,Nord-Trøndelag,A214-86-682,12.03.2026,2021,"320,000 km"
3,21140282,MAN,"€34,823 \n ...",Norway,TGX 33.500 6x4 Tractor Unit,used,Akershus,A211-40-282,12.03.2026,2017,"132,225 km"
4,21508207,MAN,"€16,950 \n ...",Vuren,18.400 TGX HYDRODRIVE,good (used),"Vuren, The Netherlands",A215-08-207,14.03.2026,2014,"688,102 km"


In [55]:
df.to_csv("tractors_dataset.csv", index=False)

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   listing_id             200 non-null    object
 1   manufacturer           200 non-null    object
 2   price                  200 non-null    object
 3   location               200 non-null    object
 4   Model:                 200 non-null    object
 5   Condition:             200 non-null    object
 6   Location:              178 non-null    object
 7   Listing ID:            200 non-null    object
 8   Update:                200 non-null    object
 9   Year of construction:  195 non-null    object
 10  Mileage:               199 non-null    object
dtypes: object(11)
memory usage: 17.3+ KB
